# Quality Stream: All HCPCS per NPI across Medicare & Medicaid (2023)

**Right Problem**: Join Medicare + Medicaid on NPI to get a unified view of all procedure codes (HCPCS/CPT), volumes, and spending per provider across both payers — scoped to **2023**.

**Datasets**:
- `MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv` — Medicare Physician & Other Practitioners, **2023**
- `medicaid-provider-spending.parquet` — Medicaid Provider Spending, 2018–2024 (**filtered to 2023**)

**Engine**: DuckDB (in-process OLAP — queries CSV/parquet directly, no loading into memory)

In [174]:
import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import os

# Persistent local DuckDB — keeps intermediate tables across cells
con = duckdb.connect("quality_stream.duckdb")

MEDICARE_FILE = "datasets/MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv"
MEDICAID_FILE = "datasets/medicaid-provider-spending.parquet"

print(f"DuckDB {duckdb.__version__} connected.")
print(f"Medicare: {os.path.getsize(MEDICARE_FILE) / 1e9:.1f} GB")
print(f"Medicaid: {os.path.getsize(MEDICAID_FILE) / 1e9:.1f} GB")

DuckDB 1.4.4 connected.
Medicare: 3.1 GB
Medicaid: 2.9 GB


---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Medicare — Schema, Shape & Sample

In [175]:
# Medicare: schema + shape
con.execute(f"CREATE OR REPLACE VIEW medicare_raw AS SELECT * FROM read_csv('{MEDICARE_FILE}', auto_detect=true)")

schema = con.execute("DESCRIBE medicare_raw").df()
shape = con.execute("SELECT count(*) AS rows FROM medicare_raw").fetchone()
print(f"Medicare shape: ({shape[0]:,} rows, {len(schema)} columns)\n")
print("Schema:")
display(schema)
print("\nSample (first 5 rows):")
con.execute("SELECT * FROM medicare_raw LIMIT 5").df()

Medicare shape: (9,660,647 rows, 28 columns)

Schema:


,column_name,column_type,null,key,default,extra
0,Rndrng_NPI,BIGINT,YES,None,None,None
1,Rndrng_Prvdr_Last_Org_Name,VARCHAR,YES,None,None,None
2,Rndrng_Prvdr_First_Name,VARCHAR,YES,None,None,None
3,Rndrng_Prvdr_MI,VARCHAR,YES,None,None,None
4,Rndrng_Prvdr_Crdntls,VARCHAR,YES,None,None,None
5,Rndrng_Prvdr_Ent_Cd,VARCHAR,YES,None,None,None
6,Rndrng_Prvdr_St1,VARCHAR,YES,None,None,None
7,Rndrng_Prvdr_St2,VARCHAR,YES,None,None,None
8,Rndrng_Prvdr_City,VARCHAR,YES,None,None,None
9,Rndrng_Prvdr_State_Abrvtn,VARCHAR,YES,None,None,None



Sample (first 5 rows):


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,HCPCS_Desc,HCPCS_Drug_Ind,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
0,1003000126,Enkeshafi,Ardalan,None,M.D.,I,6410 Rockledge Dr Ste 304,None,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,12,12.0,12,250.226667,89.062500,60.312500,54.669167
1,1003000126,Enkeshafi,Ardalan,None,M.D.,I,6410 Rockledge Dr Ste 304,None,Bethesda,MD,...,Initial hospital care with straightforward or ...,N,F,22,22.0,22,318.581818,130.312727,99.380000,98.429545
2,1003000126,Enkeshafi,Ardalan,None,M.D.,I,6410 Rockledge Dr Ste 304,None,Bethesda,MD,...,Subsequent hospital care with straightforward ...,N,F,76,127.0,127,95.732283,54.820157,43.557323,38.748661
3,1003000126,Enkeshafi,Ardalan,None,M.D.,I,6410 Rockledge Dr Ste 304,None,Bethesda,MD,...,Subsequent hospital care with moderate levelof...,N,F,180,341.0,341,194.441525,86.840762,69.086422,61.812258
4,1003000126,Enkeshafi,Ardalan,None,M.D.,I,6410 Rockledge Dr Ste 304,None,Bethesda,MD,...,Subsequent hospital care with moderate levelof...,N,F,53,79.0,79,251.454051,127.915190,100.943165,92.409114


### 1.2 Medicaid — Schema, Shape & Sample

In [176]:
# Medicaid: schema + shape
con.execute(f"CREATE OR REPLACE VIEW medicaid_raw AS SELECT * FROM read_parquet('{MEDICAID_FILE}')")

schema_med = con.execute("DESCRIBE medicaid_raw").df()
shape_med = con.execute("SELECT count(*) AS rows FROM medicaid_raw").fetchone()
print(f"Medicaid shape: ({shape_med[0]:,} rows, {len(schema_med)} columns)\n")
print("Schema:")
display(schema_med)
print("\nSample (first 5 rows):")
con.execute("SELECT * FROM medicaid_raw LIMIT 5").df()

Medicaid shape: (227,083,361 rows, 7 columns)

Schema:


,column_name,column_type,null,key,default,extra
0,BILLING_PROVIDER_NPI_NUM,VARCHAR,YES,None,None,None
1,SERVICING_PROVIDER_NPI_NUM,VARCHAR,YES,None,None,None
2,HCPCS_CODE,VARCHAR,YES,None,None,None
3,CLAIM_FROM_MONTH,VARCHAR,YES,None,None,None
4,TOTAL_UNIQUE_BENEFICIARIES,BIGINT,YES,None,None,None
5,TOTAL_CLAIMS,BIGINT,YES,None,None,None
6,TOTAL_PAID,DOUBLE,YES,None,None,None



Sample (first 5 rows):


,BILLING_PROVIDER_NPI_NUM,SERVICING_PROVIDER_NPI_NUM,HCPCS_CODE,CLAIM_FROM_MONTH,TOTAL_UNIQUE_BENEFICIARIES,TOTAL_CLAIMS,TOTAL_PAID
0,1376609297,1376609297,T1019,2024-07,39765,1205701,1.188877e+08
1,1376609297,1376609297,T1019,2024-08,39677,1152534,1.155611e+08
2,1376609297,1376609297,T1019,2024-05,39678,1157235,1.128233e+08
3,1376609297,1376609297,T1019,2024-06,39834,1164582,1.114492e+08
4,1376609297,1376609297,T1019,2024-09,39527,1099808,1.111998e+08


### 1.3 Null Values Analysis

In [177]:
# Medicare: null proportions — single SQL pass
medicare_cols = con.execute("SELECT column_name FROM (DESCRIBE medicare_raw)").df()["column_name"].tolist()
null_exprs = ", ".join([f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 4) AS \"{c}\"" for c in medicare_cols])
nulls_wide = con.execute(f"SELECT {null_exprs} FROM medicare_raw").df()

medicare_nulls = nulls_wide.T.reset_index()
medicare_nulls.columns = ["column", "null_pct"]
medicare_nulls = medicare_nulls.sort_values("null_pct", ascending=False).reset_index(drop=True)
print("Medicare — Null % per column:")
medicare_nulls

Medicare — Null % per column:


,column,null_pct
0,Rndrng_Prvdr_St2,76.1175
1,Rndrng_Prvdr_MI,35.0816
2,Rndrng_Prvdr_Crdntls,11.1841
3,Rndrng_Prvdr_First_Name,5.5595
4,Rndrng_Prvdr_RUCA_Desc,0.0787
5,Rndrng_Prvdr_RUCA,0.0787
6,Rndrng_Prvdr_State_FIPS,0.0001
7,Rndrng_NPI,0.0000
8,Tot_Srvcs,0.0000
9,Place_Of_Srvc,0.0000


In [178]:
# Medicaid: null proportions — single SQL pass
medicaid_cols = con.execute("SELECT column_name FROM (DESCRIBE medicaid_raw)").df()["column_name"].tolist()
null_exprs_med = ", ".join([f"round(100.0 * count(*) FILTER (WHERE \"{c}\" IS NULL) / count(*), 4) AS \"{c}\"" for c in medicaid_cols])
nulls_wide_med = con.execute(f"SELECT {null_exprs_med} FROM medicaid_raw").df()

medicaid_nulls = nulls_wide_med.T.reset_index()
medicaid_nulls.columns = ["column", "null_pct"]
medicaid_nulls = medicaid_nulls.sort_values("null_pct", ascending=False).reset_index(drop=True)
print("Medicaid — Null % per column:")
medicaid_nulls

Medicaid — Null % per column:


,column,null_pct
0,SERVICING_PROVIDER_NPI_NUM,4.1792
1,BILLING_PROVIDER_NPI_NUM,0.0000
2,HCPCS_CODE,0.0000
3,CLAIM_FROM_MONTH,0.0000
4,TOTAL_UNIQUE_BENEFICIARIES,0.0000
5,TOTAL_CLAIMS,0.0000
6,TOTAL_PAID,0.0000


### 1.4 Data Types Summary

In [179]:
# Side-by-side data types
mc_types = con.execute("SELECT column_name, column_type FROM (DESCRIBE medicare_raw)").df()
mc_types.columns = ["Column", "Type"]
md_types = con.execute("SELECT column_name, column_type FROM (DESCRIBE medicaid_raw)").df()
md_types.columns = ["Column", "Type"]

print("MEDICARE Data Types")
print("=" * 50)
display(mc_types)
print("\nMEDICAID Data Types")
print("=" * 50)
display(md_types)

MEDICARE Data Types


,Column,Type
0,Rndrng_NPI,BIGINT
1,Rndrng_Prvdr_Last_Org_Name,VARCHAR
2,Rndrng_Prvdr_First_Name,VARCHAR
3,Rndrng_Prvdr_MI,VARCHAR
4,Rndrng_Prvdr_Crdntls,VARCHAR
5,Rndrng_Prvdr_Ent_Cd,VARCHAR
6,Rndrng_Prvdr_St1,VARCHAR
7,Rndrng_Prvdr_St2,VARCHAR
8,Rndrng_Prvdr_City,VARCHAR
9,Rndrng_Prvdr_State_Abrvtn,VARCHAR



MEDICAID Data Types


,Column,Type
0,BILLING_PROVIDER_NPI_NUM,VARCHAR
1,SERVICING_PROVIDER_NPI_NUM,VARCHAR
2,HCPCS_CODE,VARCHAR
3,CLAIM_FROM_MONTH,VARCHAR
4,TOTAL_UNIQUE_BENEFICIARIES,BIGINT
5,TOTAL_CLAIMS,BIGINT
6,TOTAL_PAID,DOUBLE


### 1.5 Charts — Getting a Sense of the Data

In [180]:
# Chart 1: Top 20 Provider Types by record count (Medicare)
pt_df = con.execute("""
    SELECT Rndrng_Prvdr_Type AS "Provider Type", count(*) AS "Record Count"
    FROM medicare_raw
    GROUP BY Rndrng_Prvdr_Type
    ORDER BY "Record Count" DESC
    LIMIT 20
""").df()

fig = px.bar(pt_df, x="Record Count", y="Provider Type", orientation="h",
             title="Medicare: Top 20 Provider Types by Record Count",
             color="Record Count", color_continuous_scale="Blues")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [181]:
# Chart 2: Top 20 HCPCS codes by total services (Medicare)
hcpcs_df = con.execute("""
    SELECT HCPCS_Cd AS "HCPCS Code", sum(Tot_Srvcs)::BIGINT AS "Total Services"
    FROM medicare_raw
    GROUP BY HCPCS_Cd
    ORDER BY "Total Services" DESC
    LIMIT 20
""").df()

fig = px.bar(hcpcs_df, x="Total Services", y="HCPCS Code", orientation="h",
             title="Medicare: Top 20 HCPCS Codes by Total Services",
             color="Total Services", color_continuous_scale="Greens")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [182]:
# Chart 3: Top 20 HCPCS codes by total claims (Medicaid)
medicaid_hcpcs_df = con.execute("""
    SELECT HCPCS_CODE AS "HCPCS Code", sum(TOTAL_CLAIMS)::BIGINT AS "Total Claims"
    FROM medicaid_raw
    GROUP BY HCPCS_CODE
    ORDER BY "Total Claims" DESC
    LIMIT 20
""").df()

fig = px.bar(medicaid_hcpcs_df, x="Total Claims", y="HCPCS Code", orientation="h",
             title="Medicaid: Top 20 HCPCS Codes by Total Claims",
             color="Total Claims", color_continuous_scale="Oranges")
fig.update_layout(yaxis={"autorange": "reversed"}, height=600)
fig.show()

In [183]:
# Chart 4: Medicare Average Payment Distribution (sampled 50K rows)
payment_df = con.execute("""
    SELECT Avg_Mdcr_Pymt_Amt
    FROM medicare_raw
    USING SAMPLE 50000
""").df()

fig = px.histogram(payment_df, x="Avg_Mdcr_Pymt_Amt", nbins=100,
                   title="Medicare: Distribution of Avg Payment Amount (sampled 50K)",
                   labels={"Avg_Mdcr_Pymt_Amt": "Avg Medicare Payment ($)"},
                   color_discrete_sequence=["#636EFA"])
fig.update_xaxes(range=[0, 500])
fig.show()

In [184]:
# Chart 5: Medicaid Total Paid Distribution (sampled 50K rows)
medicaid_paid_df = con.execute("""
    SELECT TOTAL_PAID
    FROM medicaid_raw
    USING SAMPLE 50000
""").df()

fig = px.histogram(medicaid_paid_df, x="TOTAL_PAID", nbins=100,
                   title="Medicaid: Distribution of Total Paid (sampled 50K)",
                   labels={"TOTAL_PAID": "Total Paid ($)"},
                   color_discrete_sequence=["#EF553B"])
fig.update_xaxes(range=[0, 50000])
fig.show()

---
## 2. Data Preprocessing & Cleaning

Harmonize NPI to VARCHAR, HCPCS to uppercase VARCHAR. Aggregate metrics per NPI-HCPCS pair for each payer. All done in SQL — no Python loops.

In [185]:
# Medicare: aggregate by NPI + HCPCS, compute total payments from avg * volume
con.execute("""
    CREATE OR REPLACE TABLE medicare_agg AS
    SELECT
        CAST(Rndrng_NPI AS VARCHAR)        AS npi,
        UPPER(TRIM(HCPCS_Cd))              AS hcpcs_code,
        -- provider info (first occurrence) — COALESCE handles NULL first names for orgs
        FIRST(TRIM(COALESCE(Rndrng_Prvdr_First_Name, '') || ' ' || Rndrng_Prvdr_Last_Org_Name)) AS provider_name,
        FIRST(Rndrng_Prvdr_Type)           AS provider_type,
        FIRST(Rndrng_Prvdr_State_Abrvtn)   AS provider_state,
        FIRST(Rndrng_Prvdr_City)           AS provider_city,
        FIRST(HCPCS_Desc)                  AS hcpcs_desc,
        -- aggregated metrics
        SUM(Tot_Benes)::BIGINT             AS medicare_total_benes,
        SUM(Tot_Srvcs)::BIGINT             AS medicare_total_srvcs,
        SUM(Tot_Bene_Day_Srvcs)::BIGINT    AS medicare_total_bene_day_srvcs,
        ROUND(SUM(Avg_Mdcr_Pymt_Amt * Tot_Srvcs), 2)   AS medicare_total_pymt,
        ROUND(SUM(Avg_Sbmtd_Chrg * Tot_Srvcs), 2)      AS medicare_total_chrg,
        ROUND(SUM(Avg_Mdcr_Alowd_Amt * Tot_Srvcs), 2)  AS medicare_total_alowd
    FROM medicare_raw
    GROUP BY 1, 2
""")

stats = con.execute("SELECT count(*) AS pairs, count(DISTINCT npi) AS npis FROM medicare_agg").fetchone()
print(f"Medicare aggregated: {stats[0]:,} NPI-HCPCS pairs from {stats[1]:,} unique NPIs")
con.execute("SELECT * FROM medicare_agg LIMIT 5").df()

Medicare aggregated: 9,318,769 NPI-HCPCS pairs from 1,175,281 unique NPIs


,npi,hcpcs_code,provider_name,provider_type,provider_state,provider_city,hcpcs_desc,medicare_total_benes,medicare_total_srvcs,medicare_total_bene_day_srvcs,medicare_total_pymt,medicare_total_chrg,medicare_total_alowd
0,1003541293,99291,Matthew Imbriani,Nurse Practitioner,NJ,Ridgewood,"Critical care, first 30-74 minutes",13,13,13,2001.87,14248.00,2512.51
1,1003542168,99213,Samantha Mccullough,Nurse Practitioner,OK,Owasso,Established patient office or other outpatient...,40,40,40,2057.91,8429.74,2854.35
2,1003542622,00811,Morgan Dunlow,Certified Registered Nurse Anesthetist (CRNA),MD,Glen Burnie,Anesthesia for other procedure on large bowel ...,20,20,20,2356.46,25748.00,2919.38
3,1003546839,99344,Tracy Cowell,Nurse Practitioner,MN,Stillwater,Residence visit for new patient with moderate ...,39,39,39,3078.79,7412.72,4509.32
4,1003547563,97162,Sean Mccarty,Physical Therapist in Private Practice,MA,Winchester,"Evaluation for physical therapy, typically 30 ...",31,36,36,2948.74,8028.00,3765.58


In [186]:
# Medicaid: aggregate by servicing NPI + HCPCS — filtered to 2023 to match Medicare
# CLAIM_FROM_MONTH format is 'YYYY-MM', Medicaid raw data spans 2018–2024
con.execute("""
    CREATE OR REPLACE TABLE medicaid_agg AS
    SELECT
        TRIM(SERVICING_PROVIDER_NPI_NUM)   AS npi,
        UPPER(TRIM(HCPCS_CODE))            AS hcpcs_code,
        SUM(TOTAL_UNIQUE_BENEFICIARIES)::BIGINT AS medicaid_total_benes,
        SUM(TOTAL_CLAIMS)::BIGINT          AS medicaid_total_claims,
        ROUND(SUM(TOTAL_PAID), 2)          AS medicaid_total_paid
    FROM medicaid_raw
    WHERE CLAIM_FROM_MONTH LIKE '2023-%'
    GROUP BY 1, 2
""")

stats_med = con.execute("SELECT count(*) AS pairs, count(DISTINCT npi) AS npis FROM medicaid_agg").fetchone()
print(f"Medicaid aggregated (2023 only): {stats_med[0]:,} NPI-HCPCS pairs from {stats_med[1]:,} unique NPIs")
con.execute("SELECT * FROM medicaid_agg LIMIT 5").df()

Medicaid aggregated (2023 only): 5,770,332 NPI-HCPCS pairs from 1,020,014 unique NPIs


,npi,hcpcs_code,medicaid_total_benes,medicaid_total_claims,medicaid_total_paid
0,1558798991,T1019,1104,21716,2438512.68
1,1780251843,H2036,890,10578,2556735.00
2,1114149531,S5125,445,12223,1985185.18
3,1538300892,H2019,3486,14327,2001033.38
4,1366462897,H0040,574,8839,1460978.50


---
## 3. Join: All HCPCS per NPI across Both Payers

In [187]:
# Full outer join on NPI + HCPCS
con.execute("""
    CREATE OR REPLACE TABLE joined AS
    SELECT
        COALESCE(mc.npi, md.npi)             AS npi,
        mc.provider_name,
        mc.provider_type,
        mc.provider_state,
        mc.provider_city,
        COALESCE(mc.hcpcs_code, md.hcpcs_code) AS hcpcs_code,
        mc.hcpcs_desc,
        CASE
            WHEN mc.npi IS NOT NULL AND md.npi IS NOT NULL THEN 'Both Payers'
            WHEN mc.npi IS NOT NULL THEN 'Medicare Only'
            ELSE 'Medicaid Only'
        END AS payer_source,
        -- Medicare metrics (0 when absent)
        COALESCE(mc.medicare_total_benes, 0)          AS medicare_total_benes,
        COALESCE(mc.medicare_total_srvcs, 0)          AS medicare_total_srvcs,
        COALESCE(mc.medicare_total_pymt, 0)           AS medicare_total_pymt,
        COALESCE(mc.medicare_total_chrg, 0)           AS medicare_total_chrg,
        COALESCE(mc.medicare_total_alowd, 0)          AS medicare_total_alowd,
        -- Medicaid metrics (0 when absent)
        COALESCE(md.medicaid_total_benes, 0)          AS medicaid_total_benes,
        COALESCE(md.medicaid_total_claims, 0)         AS medicaid_total_claims,
        COALESCE(md.medicaid_total_paid, 0)           AS medicaid_total_paid
    FROM medicare_agg mc
    FULL OUTER JOIN medicaid_agg md
        ON mc.npi = md.npi AND mc.hcpcs_code = md.hcpcs_code
""")

# Summary
summary = con.execute("""
    SELECT
        count(*)              AS total_rows,
        count(DISTINCT npi)   AS unique_npis,
        count(DISTINCT hcpcs_code) AS unique_hcpcs
    FROM joined
""").fetchone()
print(f"Joined dataset: {summary[0]:,} rows | {summary[1]:,} unique NPIs | {summary[2]:,} unique HCPCS codes")

print("\nPayer source breakdown:")
display(con.execute("""
    SELECT payer_source, count(*) AS count
    FROM joined
    GROUP BY payer_source
    ORDER BY count DESC
""").df())

print("\nSample:")
con.execute("SELECT * FROM joined LIMIT 10").df()

Joined dataset: 13,573,751 rows | 1,649,487 unique NPIs | 10,743 unique HCPCS codes

Payer source breakdown:


,payer_source,count
0,Medicare Only,7803419
1,Medicaid Only,4254982
2,Both Payers,1515350



Sample:


,npi,provider_name,provider_type,provider_state,provider_city,hcpcs_code,hcpcs_desc,payer_source,medicare_total_benes,medicare_total_srvcs,medicare_total_pymt,medicare_total_chrg,medicare_total_alowd,medicaid_total_benes,medicaid_total_claims,medicaid_total_paid
0,1255524435,Derek Watson,Physical Medicine and Rehabilitation,NC,Raleigh,Q9966,"Low osmolar contrast material, 200-299 mg/ml i...",Both Payers,266,2027,699.24,2533.75,876.65,269,504,185.08
1,1255524799,Harmanjit Singh,Nephrology,CA,Torrance,99233,Subsequent hospital care with moderate levelof...,Both Payers,98,224,22021.51,38080.00,28033.16,36,79,3420.32
2,1255524930,Catalina Draghici,Psychiatry,WA,Edmonds,99231,Subsequent hospital care with straightforward ...,Both Payers,12,151,5861.63,16459.00,7357.25,12,17,237.41
3,1255525218,Air Evac Ems Inc.,Ambulance Service Provider,AR,De Queen,A0436,"Rotary wing air mileage, per statute mile",Both Payers,133,10801,337634.06,3920325.96,423765.25,42,46,72281.95
4,1255531356,Morgan White,Physician Assistant,NM,Albuquerque,99284,Emergency department visit with moderate level...,Both Payers,26,26,1913.71,23140.00,2663.71,108,114,16530.85
5,1255532099,Michael Olds,Physician Assistant,MD,Baltimore,87811,Detection test by immunoassay with direct visu...,Both Payers,72,73,2960.15,5475.00,2960.15,380,408,10948.56
6,1255534160,Richard Rupp,Diagnostic Radiology,PA,Pittsburgh,71045,"X-ray of chest, 1 view",Both Payers,623,918,6695.87,42228.00,9258.25,1188,1268,8341.14
7,1255535613,Ravis Curry,Pulmonary Disease,TN,Germantown,99232,Subsequent hospital care with moderate levelof...,Both Payers,133,218,12588.41,28340.00,15860.42,12,28,609.13
8,1255535779,Matous Pradny,Hospitalist,IN,Mishawaka,99223,Initial hospital care with moderate level of m...,Both Payers,643,719,88643.59,998691.00,116569.44,614,902,52830.11
9,1255543609,Robert Rix,Emergency Medicine,IL,Chicago,99213,Established patient office or other outpatient...,Both Payers,18,18,1331.70,4218.00,1644.67,18,18,697.44


In [188]:
# Chart 6: Payer source breakdown — pie chart
source_df = con.execute("""
    SELECT payer_source AS "Payer Source", count(*) AS "NPI-HCPCS Pairs"
    FROM joined
    GROUP BY payer_source
""").df()

fig = px.pie(source_df, values="NPI-HCPCS Pairs", names="Payer Source",
             title="NPI-HCPCS Pair Coverage: Medicare vs Medicaid Overlap",
             color="Payer Source",
             color_discrete_map={"Medicare Only": "#636EFA", "Medicaid Only": "#EF553B", "Both Payers": "#00CC96"})
fig.show()

---
## 4. Proof: Query NPI to Get All HCPCS Codes across Both Payers

In [189]:
# Proof: Dr. Brandon M. Lingenfelter, DO, PhD — OB-GYN, Princeton, WV
# Individual NPI: 1790045821
demo_npi = "1790045821"

# Provider header
info = con.execute("""
    SELECT provider_name, provider_type, provider_state, provider_city
    FROM joined
    WHERE npi = ? AND provider_name IS NOT NULL AND provider_name != ''
    LIMIT 1
""", [demo_npi]).fetchone()

if info:
    print(f"Provider: {info[0]}")
    print(f"NPI: {demo_npi}")
    print(f"Type: {info[1]} | Location: {info[3]}, {info[2]}")
else:
    print(f"NPI: {demo_npi}")
print(f"Time Range: 2023 (Medicare 2023 + Medicaid filtered to 2023)")
print("=" * 80)

# Breakdown
breakdown = con.execute("""
    SELECT
        count(*) AS total_hcpcs,
        count(*) FILTER (WHERE payer_source = 'Medicare Only') AS medicare_only,
        count(*) FILTER (WHERE payer_source = 'Medicaid Only') AS medicaid_only,
        count(*) FILTER (WHERE payer_source = 'Both Payers')   AS both_payers,
        sum(medicare_total_pymt) AS total_medicare_pymt,
        sum(medicaid_total_paid) AS total_medicaid_paid
    FROM joined
    WHERE npi = ?
""", [demo_npi]).fetchone()

print(f"\nTotal unique HCPCS codes: {breakdown[0]}")
print(f"  - Medicare only: {breakdown[1]}")
print(f"  - Medicaid only: {breakdown[2]}")
print(f"  - Both payers:   {breakdown[3]}")
print(f"\nMedicare total payment: ${breakdown[4]:,.2f}")
print(f"Medicaid total paid:    ${breakdown[5]:,.2f}")

print(f"\nAll HCPCS codes for this provider:")
con.execute("""
    SELECT hcpcs_code, hcpcs_desc, payer_source,
           medicare_total_srvcs, medicare_total_pymt,
           medicaid_total_claims, medicaid_total_paid
    FROM joined
    WHERE npi = ?
    ORDER BY hcpcs_code
""", [demo_npi]).df()

Provider: Brandon Lingenfelter
NPI: 1790045821
Type: Obstetrics & Gynecology | Location: Princeton, WV
Time Range: 2023 (Medicare 2023 + Medicaid filtered to 2023)

Total unique HCPCS codes: 33
  - Medicare only: 4
  - Medicaid only: 21
  - Both payers:   8

Medicare total payment: $43,475.08
Medicaid total paid:    $74,136.77

All HCPCS codes for this provider:


,hcpcs_code,hcpcs_desc,payer_source,medicare_total_srvcs,medicare_total_pymt,medicaid_total_claims,medicaid_total_paid
0,36415,Insertion of needle into vein for collection o...,Both Payers,39,319.20,398,1328.56
1,51725,Simple measurement of pressure of urine flow i...,Medicare Only,18,2247.89,0,0.00
2,58558,Biopsy of lining of uterus and/or removal of p...,Medicare Only,17,2797.49,0,0.00
3,59025,None,Medicaid Only,0,0.00,62,957.26
4,76816,None,Medicaid Only,0,0.00,330,22226.45
5,76830,"Ultrasound scan of uterus, ovaries, tubes, cer...",Both Payers,42,3338.01,30,1561.52
6,80048,None,Medicaid Only,0,0.00,95,717.66
7,81001,None,Medicaid Only,0,0.00,89,235.96
8,81002,"Urinalysis, manual test",Both Payers,51,167.86,631,1232.68
9,81025,None,Medicaid Only,0,0.00,85,573.06


In [190]:
# Reusable lookup function — query any NPI via DuckDB
def lookup_npi(npi_str):
    """Query all HCPCS codes for a given NPI across both payers."""
    npi_str = str(npi_str).strip()
    result = con.execute("""
        SELECT hcpcs_code, hcpcs_desc, payer_source,
               medicare_total_srvcs, medicare_total_pymt,
               medicaid_total_claims, medicaid_total_paid
        FROM joined
        WHERE npi = ?
        ORDER BY hcpcs_code
    """, [npi_str]).df()
    if len(result) == 0:
        print(f"No records found for NPI: {npi_str}")
        return None
    info = con.execute("""
        SELECT provider_name, provider_type, provider_state
        FROM joined WHERE npi = ? AND provider_name IS NOT NULL AND provider_name != '' LIMIT 1
    """, [npi_str]).fetchone()
    if info:
        print(f"Provider: {info[0]} | Type: {info[1]} | State: {info[2]}")
    print(f"NPI: {npi_str} | Total HCPCS codes: {len(result)}")
    mc_only = (result["payer_source"] == "Medicare Only").sum()
    md_only = (result["payer_source"] == "Medicaid Only").sum()
    both = (result["payer_source"] == "Both Payers").sum()
    print(f"  Medicare only: {mc_only} | Medicaid only: {md_only} | Both: {both}")
    return result

# Example: Dr. Brandon M. Lingenfelter, DO, PhD — OB-GYN, Princeton, WV
lookup_npi("1790045821")

Provider: Brandon Lingenfelter | Type: Obstetrics & Gynecology | State: WV
NPI: 1790045821 | Total HCPCS codes: 33
  Medicare only: 4 | Medicaid only: 21 | Both: 8


,hcpcs_code,hcpcs_desc,payer_source,medicare_total_srvcs,medicare_total_pymt,medicaid_total_claims,medicaid_total_paid
0,36415,Insertion of needle into vein for collection o...,Both Payers,39,319.20,398,1328.56
1,51725,Simple measurement of pressure of urine flow i...,Medicare Only,18,2247.89,0,0.00
2,58558,Biopsy of lining of uterus and/or removal of p...,Medicare Only,17,2797.49,0,0.00
3,59025,None,Medicaid Only,0,0.00,62,957.26
4,76816,None,Medicaid Only,0,0.00,330,22226.45
5,76830,"Ultrasound scan of uterus, ovaries, tubes, cer...",Both Payers,42,3338.01,30,1561.52
6,80048,None,Medicaid Only,0,0.00,95,717.66
7,81001,None,Medicaid Only,0,0.00,89,235.96
8,81002,"Urinalysis, manual test",Both Payers,51,167.86,631,1232.68
9,81025,None,Medicaid Only,0,0.00,85,573.06
